# M12 Comprehensive Experimental Lab: Combine-to-NFL movement transfer

**NFL Big Data Bowl 2027 · Representation learning and NFL transfer research laboratory · Google Colab GPU**

This notebook only orchestrates. Every algorithm lives in the `nflbowl` package (`nflbowl.lab`, `nflbowl.Experiment`) and in the unchanged Milestone 12 research code (`nflbdb...m12`). Every displayed result is saved to Google Drive with its data, and every section records its status in a stage ledger.

> **No result in this notebook is pre-filled.** The tables, figures and interpretation sentences come from computations run in this session, or from artifacts reloaded from Drive and identified by their bundle fingerprint and science hash. Earlier exploratory numbers are quoted only as hypotheses to test.

---

## A1 · Executive research overview

**Competition objective.** Find non-obvious, actionable relationships between high-frequency Combine tracking and later NFL movement.

**Why stopwatch tests are not enough.** A forty time summarises a whole acceleration curve in one number. Two athletes can share a 4.50 and still differ in how quickly they build speed, how hard they brake, how tightly they turn at speed, and how consistent they are from attempt to attempt. 10 Hz tracking measures these qualities directly.

**Hypotheses, to be tested and not assumed:**

| # | Hypothesis | Where it is tested |
|---|---|---|
| H-a | Tracking quantifies movement qualities that stopwatch metrics miss | Part H |
| H-b | An encoder learns an athlete-specific representation shared by Combine and NFL contexts | Parts C, D |
| H-c | The representation is robust to domain shift | Parts C, G |
| H-d | It generalises to athletes excluded from training (inductive) | Part D |
| H-e | It improves out-of-fold prediction of NFL movement capacity beyond M1 and M1+F0 | Part E |
| H-f | Gains are consistent across positions and athletic profiles | Parts E, G |
| H-g | Scouts can use some of the signals | Parts H, I |
| H-h | Apparent gains are artefacts of leakage, imbalance or construction | Parts B, G |

**Prior exploratory F4 results (hypotheses only, not inputs).** Transductive retrieval AUC was about 0.497 for F0, 0.500 for the random encoder and 0.524 for F4. Domain AUC was about 0.99 for all three, and MMD² was about 0.20 / 0.25 / 0.12. These figures raise four questions. Is retrieval above chance at all? Does a lower MMD² hide domain information that is still present? Does transductive retrieval generalise? Do better embeddings mean better NFL prediction?

**Regimes.** *Transductive*: an encoder trained on every sequence, used only as a label-free diagnostic. *Inductive*: one encoder per outer fold, trained without any Combine or NFL window of that fold's athletes, and evaluated on those athletes. The two are never pooled.

**What would count as actionable scouting evidence.** All of the following must hold. Gate 1 (data and leakage) passes. Inductive retrieval exceeds both chance and F0, with an interval, on all folds. Out-of-fold ΔRMSE beats both M1 and M1+F0, with an interval. The result survives the robustness checks. The signal is carried by repeatable, physically defined movement metrics. Part J evaluates these gates by declared rules.

```text
 Bundle (77,125 × 64 × 6, 510 athletes, 5 athlete folds, fingerprint 4867be08f8191b3a)
   │
   ├─ B  audit · composition · attrition · leakage ──────────────────────────────┐
   │                                                                             │
   ├─ D2 inductive training (fold f encoder never sees fold f athletes) ─┐       │
   │                                                                     ▼       ▼
   ├─ C  geometry / collapse / domain (held-out rows) ◄──────── embeddings ──► D retrieval (paired, identical pools)
   │                                                                     │
   ├─ H  biomechanical metrics (deterministic) ──┐                       ▼
   │                                             ├──► E  NFL transfer ladder (OOF, frozen nfl.q90_shared)
   │                                             └──► I  interpretability · athlete profiles
   ├─ F  ablations (registry)   G  seeds · pools · quality · heterogeneity
   └─ J  gates + integrated comparison ──► K  final report (from saved artifacts only)
```

### How to run

| Phase | Sections | Compute | Needs |
|---|---|---|---|
| 1 Setup and audit | Setup, A, B | CPU, a few minutes | the cleaned bundle |
| 2 Cached baselines | checkpoint adoption, D1 | CPU | earlier experiments on Drive (optional) |
| 3 GPU training | D2 (and F, G1, G2 when enabled) | GPU, heavy | `run_training=True` |
| 4 Embedding extraction | inside C, D | GPU, light | trained or adopted checkpoints |
| 5 CPU evaluation | C, D, E, G, H | CPU | embeddings |
| 6 Interpretation | I, J | CPU | saved tables |
| 7 Reporting | K | CPU | anything saved |

**Run modes** (`CFG['run_mode']`):
- `pilot`: the selected held-out fold and the selected encoder only, plus the F0 and random references.
- `standard`: F2 and F4 on all five folds.
- `full`: also enables the secondary encoders and, behind their own flags, ablations, seeds and learning curves.

Expensive work is never switched on implicitly.

**Resuming.** Re-run Setup 1–3 with the same `experiment_name`. Training resumes from the last verified epoch checkpoint, and every section reloads its saved artifacts when the bundle fingerprint and science hash match. A failing section is isolated: the ledger records `failed`, the traceback is written to `logs/`, and later independent sections still run.

**Recommended order.** Pilot first, with `run_training=False`, to validate the setup and audit. Then pilot with `run_training=True`. Then `standard` with training and transfer. `full` comes last, and only once the primary comparison is stable. Pin `GIT_REF` to the tested commit SHA before the final multi-fold run.

In [ ]:
# Setup 1: install nflbowl from GitHub. Colab's CUDA build of PyTorch is never replaced, because torch is not a dependency.
# Before the final multi-fold run, pin GIT_REF to the exact tested commit SHA, e.g. GIT_REF = '0123abcd...'.
GIT_REF = 'main'
%pip install -q 'git+https://github.com/mybenkhadda/NFL_bowl.git@{GIT_REF}'
%pip install -q --upgrade --force-reinstall --no-deps 'git+https://github.com/mybenkhadda/NFL_bowl.git@{GIT_REF}'
%pip install -q umap-learn markdown
# If nflbowl was already imported in this runtime: Runtime > Restart session, then continue with Setup 2.

In [ ]:
# Setup 2: imports, Drive, configuration (validated: unknown keys raise), CUDA check
import json
from pathlib import Path

import numpy as np
import polars as pl

import nflbowl
from nflbowl.data.validation import TRANSFER_TARGET
from nflbowl.lab import ablations, audit, biomech, environment, geometry, interpret, registry, retrieval_lab, robustness, stats, synthesis, transfer_lab
from nflbowl.lab import config as labconfig
from nflbowl.lab import figures as F
from nflbowl.lab.store import LabRun, StageSkipped
from nflbowl.models import registry as reg
from nflbowl.research import colab
from nflbowl.research.artifacts import markdown_to_html

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(90)
print('nflbowl', nflbowl.__version__)
colab.assert_version('0.4.0')
print('Drive mounted:', colab.mount_drive())

CFG = {
    'experiment_name': 'm12_comprehensive_lab_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'device': 'auto',
    'precision': 'fp32',            # fp32 default; AMP only through the library's fixed losses (never a notebook patch)
    'seed': 42,                     # LAB seed (figure sampling, case studies); the science seed stays that of the base preset
    'resume': True,
    'run_mode': 'pilot',            # 'pilot' | 'standard' | 'full'
    'run_training': False,
    'run_inductive': True,
    'run_transfer': False,
    'run_ablations': False,
    'run_full_diagnostics': True,
    'reuse_artifacts': True,
    'save_figures': True,
    'save_tables': True,
    'save_processed_data': True,
    'save_reports': True,
    'save_checkpoints': True,
    # optional (defaults shown in nflbowl/lab/config.py): pilot_fold, pilot_encoder, primary_encoders, secondary_encoders,
    # reference_experiments, import_checkpoints, run_seeds, run_learning_curves, run_permutation_tests, n_boot, science_seed
}
cfg = labconfig.resolve({**CFG, 'git_ref': globals().get('GIT_REF', 'main')})
DRIVE_ROOT = Path(cfg.drive_root)
EXPERIMENT_NAME = cfg.experiment_name
RUN_DIR = DRIVE_ROOT / 'experimental_lab' / EXPERIMENT_NAME
GPU = colab.gpu_memory_summary()
print('GPU:', GPU)
if not GPU.get('cuda'):
    print('WARNING: no CUDA GPU. Analysis sections run on CPU; training (run_training=True) would be very slow.')
print('RUN_DIR:', RUN_DIR)

In [ ]:
# Setup 3: run directory, environment record, bundle validation (read-only), lab experiment, precision lock
run = LabRun(RUN_DIR, cfg, local_tmp='/content/lab_tmp')
R = run.results
env = environment.environment_record(cfg)
if cfg.precision in ('fp16', 'auto') and not env['fp16_loss_fix'].get('ok'):
    raise RuntimeError('mixed precision requested but the installed library lacks the fp32 contrastive-loss fix: reinstall (Setup 1) or use fp32')

def _setup(ctx):
    checks, summary = environment.validate_bundle(cfg)
    run.table(checks, 'bundle_checks', 'dataset')
    R['bundle_checks'], R['bundle_summary'] = checks, summary
    if checks.filter(pl.col('status') == 'fail').height:
        display(checks)
        raise RuntimeError('the bundle does not match its declared provenance: fix the bundle path before running anything else')
    return checks

BUNDLE_CHECKS = run.run_stage('SETUP', _setup, note='bundle validation', inputs=[cfg.bundle_dir], raise_errors=True)
exp = environment.open_experiment(cfg, run)
LOCK = environment.precision_lock(run, cfg, exp)
run.write_config(cfg, {**env, 'experiment': exp.id, 'experiment_root': str(exp.root), 'science_hash': exp.cfg.science_hash(),
                       'science': exp.cfg.to_dict()['science'], 'bundle_fingerprint': exp.bundle.fingerprint, 'precision_lock': LOCK})
F0N, RANDN, F2N, F4N = reg.resolve('F0'), reg.resolve('random'), reg.resolve('F2'), reg.resolve('F4')
ALL_FOLDS = sorted(set(exp.folds_by_athlete.values()))
FOLDS = cfg.folds(ALL_FOLDS)
ENCODERS = [reg.resolve(m) for m in cfg.encoders()]
MODELS = [F0N, RANDN, *ENCODERS]
SALT = cfg.experiment_name
PSEUDO = lambda i: stats.pseudonym(i, SALT) if cfg.anonymize_ids else str(i)
display(BUNDLE_CHECKS)
display(environment.setup_summary(env, BUNDLE_CHECKS, exp, cfg))
print('folds evaluated:', FOLDS, '| encoders scheduled:', ENCODERS)

## A2 · Reproducibility and scientific controls

**Research question:** can every number in this run be traced to exact inputs?

The record covers the bundle fingerprint, the science configuration hash (part of every checkpoint and artifact signature), the seeds (science and lab), hardware, precision, library versions, fold definitions, model configurations, the training schedule and the evaluation settings. It is saved to `config/run_config.json` and `config/environment.json`.

The lab seed (`CFG['seed']`) only drives figure sampling and case-study draws. The science seed is the one in the nflbowl preset, which is what lets compatible earlier checkpoints be reused. Changing it (`science_seed`) creates a different science hash, and nothing is shared across it.

In [ ]:
def _a2(ctx):
    sci = exp.cfg.to_dict()
    rows = [('bundle fingerprint', exp.bundle.fingerprint), ('bundle dir', cfg.bundle_dir), ('science hash', exp.cfg.science_hash()),
            ('science seed', sci['science']['seed']), ('lab seed', cfg.seed), ('precision (runtime)', cfg.precision),
            ('device', env['gpu'].get('device') if env['gpu'].get('cuda') else 'cpu'), ('code version', json.dumps(env['code_version'])),
            ('git ref requested', cfg.git_ref), ('outer folds', str(ALL_FOLDS)), ('val share (athletes)', sci['science']['val_share']),
            ('epochs / patience / batch', f"{sci['science']['epochs']} / {sci['science']['patience']} / {sci['science']['batch']}"),
            ('embed dim / width', f"{sci['science']['embed_dim']} / {sci['science']['width']}"), ('bootstrap (lab)', cfg.boot),
            ('inductive configs (science)', ', '.join(sci['science']['inductive_configs'])), ('evaluation backend', sci['evaluation']['backend'])]
    t = pl.DataFrame({'item': [r[0] for r in rows], 'value': [str(r[1]) for r in rows]})
    run.table(t, 'reproducibility_record', 'synthesis')
    vers = pl.DataFrame({'package': list(env['versions']), 'version': [str(v) for v in env['versions'].values()]})
    run.table(vers, 'library_versions', 'synthesis')
    folds = exp.bundle.folds.join(pl.DataFrame({'nfl_id': list(exp.folds_by_athlete), 'fold_used': list(exp.folds_by_athlete.values())}), on='nfl_id', how='full', coalesce=True)
    run.table(folds, 'fold_definitions', 'dataset')
    run.table(reg.list_models(), 'model_registry', 'ablations')
    display(t)
    return t

run.run_stage('A2', _a2, note='reproducibility record')

## A3 · Experiment registry

Each planned experiment declares its hypothesis, inputs, representation, regime, held-out protocol, baseline, metrics, statistics, outputs, compute class, gating flag, interpretation rule and feasibility on the portable bundle (see `nflbowl/lab/registry.py`).

Status is read from the stage ledger, never inferred: `completed` · `cached` · `pending` · `skipped` · `blocked` · `failed`. The table is re-exported at the end of the run (Part K).

In [ ]:
STATUS = registry.status_table(run)
run.save_table(registry.specs_frame(), 'experiment_registry_specs', subdir='config')
run.save_table(STATUS, 'experiment_registry', subdir='config')
run.write_text(registry.matrix_markdown(STATUS), 'config/experiment_registry.md')
display(STATUS.select('id', 'title', 'status', 'compute', 'modes', 'flag', 'feasibility'))

### Phase 2 · Compatible checkpoints from earlier experiments (read-only)

Earlier experiments (`f4_main_clean`, `f4_main`, unified runs) are scanned through their JSON manifests only, and are never modified.

An inductive checkpoint is **adopted** (copied into this lab's experiment and registered with its original dependency record) only when its bundle fingerprint **and** science hash equal this lab's. Anything else is listed with the reason it is incompatible. `f4_main` was trained on the uncleaned bundle, so it is expected to be incompatible.

In [ ]:
def _adopt(ctx):
    src = environment.discover_sources(cfg, exp)
    run.data_file(src, 'diagnostics/checkpoint_sources', {'bundle': exp.bundle.fingerprint, 'science': exp.cfg.science_hash()})
    if src.height and 'status' in src.columns:
        display(src.group_by('experiment', 'status').agg(pl.len().alias('records')).sort('experiment'))
    if not (cfg.import_checkpoints and cfg.reuse_artifacts):
        raise StageSkipped('import_checkpoints / reuse_artifacts disabled')
    adopted = environment.adopt_checkpoints(exp, src, models=None, regime='inductive')
    run.data_file(adopted, 'diagnostics/checkpoints_adopted', {'experiment': exp.id})
    ctx['cache'] = 'hit' if adopted.height else None
    return adopted

ADOPTED = run.run_stage('P2', _adopt, note='discover / adopt compatible inductive checkpoints')
if ADOPTED is not None:
    display(ADOPTED)
INDUCTIVE_STATUS = environment.inductive_status(exp, [*ENCODERS, *(ablations.ablation_names() if cfg.run_mode == 'full' else [])], FOLDS)
run.table(INDUCTIVE_STATUS, 'inductive_status_before_training', 'ablations')
display(INDUCTIVE_STATUS)

# PART B · Advanced dataset characterization

### Research question
Does the cleaned bundle match its declared provenance? How do the Combine and NFL domains differ, which athletes enter each analysis, and where could leakage or shortcuts enter?

### Scientific rationale
Every later comparison is conditional on the sample. Domain differences bound what *invariance* can mean. Attrition that is not random across positions changes what a pooled estimate represents.

### Data and methods
All counts are exact and computed on the whole bundle. Sequence-level statistics are summarised with an **athlete-cluster bootstrap**: athletes are resampled together with all of their sequences, never as independent rows. Per-sequence physical metrics are the deterministic Part H definitions (no fitting), cached to `datasets/cached/sequence_metrics.parquet` with the bundle fingerprint and the metric version.

In [ ]:
# B1: dataset audit
def _b1(ctx):
    global SEQ, ATH
    SEQ = biomech.sequence_table(exp, run, reuse=cfg.reuse_artifacts)
    ATH = audit.athlete_table(exp, SEQ)
    run.data_file(ATH, 'datasets/cached/athlete_table', {'bundle': exp.bundle.fingerprint})
    T = audit.dataset_overview(exp, SEQ, ATH)
    for k, v in T.items():
        if v is not None and v.height:
            run.table(v, k, 'dataset')
    R['dataset_overview'] = T['overview']
    display(T['overview'])
    display(T['valid_frames'])
    pac = T['per_athlete_long'].with_columns(pl.col('domain').replace({'n_combine_seq': 'combine', 'n_nfl_seq': 'nfl'}))
    run.figure(F.distribution_compare(pac, 'n_seq', 'domain', 'Sequences per athlete by domain', 'sequences per athlete', facet='position_group'),
               'b1_sequences_per_athlete', {'per_athlete': pac}, 'How unevenly are athletes represented in each domain?',
               'Box (IQR) + violin per position and domain; right: pooled ECDF.')
    run.figure(F.histogram_by(SEQ.select('domain', 'n_valid'), 'n_valid', 'domain', 'Valid frames per sequence', 'valid frames (10 Hz)', bins=64),
               'b1_valid_frames', {'valid_frames': T['valid_frames']}, 'Do Combine and NFL sequences have comparable lengths?', 'Step histograms by domain.')
    run.figure(F.heatmap(T['position_domain'], 'position_group', 'domain', 'n_athletes', 'Athletes per position group and domain', fmt='{:,.0f}'),
               'b1_position_domain_heatmap', {'position_domain': T['position_domain']}, 'Which positions are thinly represented?', 'Athlete counts.')
    miss = T['missingness_athletes']
    if miss.height:
        ml = miss.drop('n_linked').unpivot(index='position_group', variable_name='variable', value_name='missing_share')
        run.figure(F.heatmap(ml, 'position_group', 'variable', 'missing_share', 'Missing share of officials and target (linked athletes)', fmt='{:.0%}', vmin=0, vmax=1),
                   'b1_missingness_heatmap', {'missingness': ml}, 'Which officials are missing, and for whom?', 'Share missing per position.')
    run.figure(F.stacked(T['fold_balance'], 'fold', 'position_group', 'n_athletes', 'Fold composition by position group', 'athletes'),
               'b1_fold_composition', {'fold_balance': T['fold_balance']}, 'Are outer folds balanced by position?', 'Athletes per fold, stacked by position.')
    if TRANSFER_TARGET in ATH.columns:
        tg = ATH.filter(pl.col(TRANSFER_TARGET).is_not_null()).select('position_group', TRANSFER_TARGET).with_columns(pl.lit('nfl').alias('series'))
        run.figure(F.distribution_compare(tg, TRANSFER_TARGET, 'series', 'Frozen NFL movement-capacity target by position', 'nfl.q90_shared', facet='position_group', max_facets=12),
                   'b1_target_by_position', {'target': tg}, 'How does the frozen target vary by position?', 'Per-position distribution of nfl.q90_shared.')
    C, cols = audit.m1_correlations(ATH)
    run.figure(F.correlation_matrix(C, cols, 'Spearman correlations: officials, target, coverage'), 'b1_feature_correlation',
               {'correlation': pl.DataFrame({'variable': cols, **{c: C[:, j] for j, c in enumerate(cols)}})}, 'Which officials are redundant, and do coverage counts track the target?',
               'Pairwise-complete Spearman correlations.')
    return T

B1 = run.run_stage('B1', _b1, note='dataset audit', inputs=['bundle'])

In [ ]:
# B2: domain composition
def _b2(ctx):
    comp, bypos = audit.domain_composition(SEQ, cfg.boot, cfg.seed)
    st, cat = audit.state_category_composition(SEQ)
    run.table(comp, 'domain_composition', 'dataset')
    run.table(bypos, 'domain_by_position', 'dataset')
    run.table(st, 'domain_state_composition', 'dataset')
    run.table(cat, 'domain_category_composition', 'dataset')
    R['domain_composition'] = comp
    display(comp.select('metric', 'median_combine', 'median_nfl', 'diff_nfl_minus_combine', 'lo', 'hi', 'smd', 'position_adjusted_diff'))
    for m, lab in (('peak_speed', 'peak speed (yd/s)'), ('peak_accel', 'peak tangential acceleration (yd/s²)'), ('peak_decel', 'peak braking (yd/s²)'),
                   ('max_abs_a_n', 'max |normal acceleration| (yd/s²)'), ('heading_rate_sd', 'heading-rate SD (rad/s)'), ('duration_s', 'duration (s)')):
        d = SEQ.select('domain', 'position_group', m)
        run.figure(F.distribution_compare(d, m, 'domain', f'Combine vs NFL · {lab}', lab, facet='position_group', order=['combine', 'nfl']),
                   f'b2_domain_{m}', {'summary': comp.filter(pl.col('metric') == m), 'by_position': bypos.filter(pl.col('metric') == m)},
                   f'How does {lab} differ between Combine drills and NFL windows, within position?', 'Per-position IQR + violin; pooled ECDF.', show=m in ('peak_speed', 'max_abs_a_n'))
    rel = bypos.with_columns((pl.col('diff') / pl.col('median_combine').abs()).alias('relative_diff'))
    run.figure(F.heatmap(rel, 'metric', 'position_group', 'relative_diff', 'NFL − Combine median difference relative to the Combine median', fmt='{:+.0%}', diverging=True),
               'b2_position_adjusted_differences', {'relative': rel}, 'Is the domain gap the same for every position?', 'Diverging scale centred on 0.')
    run.figure(F.stacked(st, 'domain', 'state', 'share', 'Movement-state composition (alignment labels)', 'share of sequences'),
               'b2_state_composition', {'state': st}, 'Do the matched-state alignment labels overlap across domains?', 'Start-speed bin × accelerating flag.')
    kinds = B1['kinds'] if B1 else SEQ.group_by('domain', 'kind').agg(pl.len().alias('n_sequences'))
    kd = kinds.with_columns((pl.col('domain') + ' · ' + pl.col('kind')).alias('label')).sort('domain', 'n_sequences', descending=[False, True])
    run.figure(F.hbar(kd, 'label', 'n_sequences', 'Drill families (Combine) and demand categories (NFL)', 'sequences', color_by='domain'),
               'b2_drill_vs_demand', {'kinds': kd}, 'Which drill families / NFL demands dominate the data?', 'Sequence counts.')
    return comp

B2 = run.run_stage('B2', _b2, note='domain composition', inputs=['B1'])

In [ ]:
# B3: sample selection, coverage, attrition
def _b3(ctx):
    steps, bypos = audit.attrition(ATH)
    run.table(steps, 'attrition', 'dataset')
    run.table(bypos, 'attrition_by_position', 'dataset')
    display(steps)
    run.figure(F.waterfall(steps, 'Athlete coverage and attrition', 'Grey = removed at each step; blue = remaining (retrieval needs both domains; transfer needs link + target).'),
               'b3_attrition_waterfall', {'steps': steps}, 'Where are athletes lost before each analysis?', 'Sequential eligibility steps.')
    last = bypos.filter(pl.col('step') == steps['step'][-1]).with_columns(
        pl.when(pl.col('n_remaining') < cfg.min_subgroup).then(pl.lit('insufficient')).when(pl.col('n_remaining') < cfg.adequate_n).then(pl.lit('exploratory'))
        .otherwise(pl.lit('adequate')).alias('reliability'))
    run.table(last, 'eligible_by_position', 'dataset')
    run.figure(F.hbar(last.sort('n_remaining', descending=True), 'position_group', 'n_remaining', 'Fully eligible athletes per position (transfer cohort)', 'athletes',
                      ref=cfg.adequate_n), 'b3_eligible_by_position', {'eligible': last}, 'Which positions can support subgroup inference?',
               f'Dashed line = adequate_n ({cfg.adequate_n}).')
    return steps

B3 = run.run_stage('B3', _b3, note='attrition', inputs=['B1'])

In [ ]:
# B4: leakage audit (runtime checks + documented preprocessing scope). Re-run after D2 to cover newly trained encoders.
def _b4(ctx):
    mf = environment.available_pairs(exp, MODELS, FOLDS)
    L = audit.leakage_audit(exp, SEQ, ATH, mf, cfg.seed)
    run.table(L['checks'], 'leakage_checks', 'dataset')
    run.table(L['shortcuts'], 'leakage_shortcuts', 'dataset')
    run.table(L['duplicates'], 'leakage_duplicates', 'dataset')
    run.table(L['scope'], 'preprocessing_scope', 'dataset')
    run.data_file(L['duplicate_groups'], 'diagnostics/duplicate_sequence_groups', {'bundle': exp.bundle.fingerprint})
    R['leakage_checks'], R['shortcuts'] = L['checks'], L['shortcuts']
    display(L['checks'].group_by('status').agg(pl.len().alias('checks')))
    display(L['checks'].filter(pl.col('status') != 'pass'))
    display(L['shortcuts'])
    display(L['scope'])
    return L

B4 = run.run_stage('B4', _b4, note='leakage audit', inputs=['B1', 'training records', 'embedding tables'])

In [ ]:
# Part B interpretation (generated from measured values) + report
lines = []
if B2 is not None:
    for r in B2.iter_rows(named=True):
        lines.append(synthesis.describe_interval(f"NFL − Combine median {r['metric']}", r['diff_nfl_minus_combine'], r['lo'], r['hi'], positive_means='higher in NFL windows'))
if B4 is not None:
    nfail = B4['checks'].filter(pl.col('status') == 'fail').height
    lines.append(f'- Leakage audit: {nfail} failing check(s). Checks verified only by code inspection are labelled so in the scope table; athlete-id separation alone is not taken as proof.')
    triv = B4['shortcuts'].filter(pl.col('reading') == 'trivial shortcut')['feature'].to_list()
    lines.append(f'- Non-movement domain shortcuts with separability ≥ 0.9: {triv or "none"}. Any representation that encodes them separates domains trivially.')
synthesis.section_report(run, 'b_dataset_characterization', 'Part B · Dataset characterization', 'Does the bundle match its provenance, how do domains differ, who is analysed, and where could leakage enter?',
                         'Every later comparison is conditional on the sample and on the domain gap.',
                         ['exact counts on the full bundle', 'athlete-cluster bootstrap for sequence statistics', 'deterministic Part H metrics'],
                         {k: R.get(k) for k in ('dataset_overview', 'domain_composition', 'leakage_checks', 'shortcuts')}, lines,
                         ['Physical units rely on the store scaling (rep.SCALE).', 'Domain gaps are expected: Combine drills and NFL windows are different tasks.'],
                         status=f"B1 {run.stage_status('B1')} · B2 {run.stage_status('B2')} · B3 {run.stage_status('B3')} · B4 {run.stage_status('B4')}")
print('\n'.join(lines))

# Phase 3 · D2 · Inductive encoder training (GPU)

### Research question
Can the encoders be trained stably without any window of the held-out athletes?

### Methods
`Experiment.train(model, regime='inductive', folds=[f])` trains one encoder per outer fold. The training rows exclude every Combine **and** NFL sequence of fold *f*. Early stopping uses a seeded share of the *other* athletes. Physics targets are standardised with training-row constants only.

Training resumes from the last verified epoch shard. A non-finite loss stops with `FloatingPointError`, which names the loss term. The precision is the locked lab precision (fp32 by default).

Only (model, fold) pairs without a training record are trained. Adopted checkpoints count as trained. Runs only when `run_training=True`.

In [ ]:
def _d2(ctx):
    t = environment.train_missing(exp, ENCODERS, FOLDS, resume=cfg.resume)
    if t.height:
        run.table(t, 'training_runs', 'ablations')
        display(t)
    return t

run.run_stage('D2', _d2, note='inductive training', inputs=['bundle'], enabled=cfg.run_training and cfg.run_inductive,
              reason_disabled='run_training=False (using adopted / existing checkpoints only)')

MODELS_FOLDS = environment.available_pairs(exp, MODELS, FOLDS)
PRIMARY_FOLDS = MODELS_FOLDS.get(F4N, [])
run.table(pl.DataFrame({'fold': PRIMARY_FOLDS}, schema={'fold': pl.Int64}), 'folds_used', 'retrieval')
CURVES = ablations.training_curves(exp)
HEALTH = ablations.training_health(CURVES)
if CURVES.height:
    run.table(CURVES, 'training_curves', 'ablations')
    run.table(HEALTH, 'training_health', 'ablations')
    display(HEALTH)
    cl = ablations.curves_long(CURVES.filter(pl.col('model').is_in(ENCODERS)))
    run.figure(F.small_multiples_lines(cl, 'component', 'epoch', 'value', 'run', 'Loss components per epoch (inductive and transductive runs)', 'epoch', 'loss',
                                       ncols=3, max_facets=9), 'd2_loss_components', {'curves': cl}, 'Did every objective term train stably?',
               'Each panel = one loss term; one line per model · fold.')
print('available (model → folds):', {F.short(k): v for k, v in MODELS_FOLDS.items()})
display(environment.inductive_status(exp, ENCODERS, FOLDS))

# PART C · Representation geometry and signal structure

### Research question
What do the embeddings encode? Are they collapsed? How far apart are Combine and NFL in each representation?

### Scientific rationale
Retrieval near chance can come from collapse, from domain-specific encoding, or from a genuine absence of athlete signal. Geometry separates these explanations.

### Data and methods
All quantities are computed on the **held-out rows** of each outer fold, embedded by the encoder that never saw those athletes. F0 and the random encoder have nothing fitted.

- Effective rank (exponential of the spectral entropy), participation ratio, PC1 share, near-constant dimensions, dimension correlation, norms, pairwise cosines, nearest-neighbour distances, k-NN domain mixing.
- Domain metrics: athlete-grouped ridge **domain AUC** (overall and within position, position-adjusted average) and **MMD²** (the Milestone 11 estimator behind the transductive values).
- Collapse rule (declared): effective rank < 2, or more than 20 % dead dimensions, or mean pairwise cosine > 0.9.

### Interpretation rule
Lower MMD² **with** domain AUC ≥ 0.9 means distribution alignment **without** invariance. Projections are exploratory pictures, never evidence.

In [ ]:
def _c(ctx):
    G = geometry.run_geometry(exp, MODELS_FOLDS, cfg.seed)
    run.table(G['summary'], 'geometry_summary', 'geometry')
    run.table(G['summary_long'], 'geometry_summary_by_fold', 'geometry')
    run.table(G['spectrum'], 'covariance_spectrum', 'geometry')
    run.table(G['pairs'], 'athlete_pair_similarity', 'geometry')
    run.table(G['domain_long'], 'domain_diagnostics_long', 'geometry')
    run.data_file(G['norms'], 'embeddings/inductive/embedding_norms', {'bundle': exp.bundle.fingerprint})
    flags = geometry.collapse_flags(G['summary'])
    run.table(flags, 'collapse_flags', 'geometry')
    keep = ['dims', 'effective_rank', 'participation_ratio', 'pc1_share', 'n_dead_dims', 'mean_pairwise_cosine', 'max_abs_dim_corr', 'knn_domain_mixing_excess',
            'domain_auc_grouped', 'position_adjusted_domain_auc', 'mmd2']
    ov = G['summary'].filter(pl.col('metric').is_in(keep)).pivot(on='metric', index='representation', values='mean')
    run.table(ov, 'geometry_overview', 'geometry')
    R.update({'geometry_summary': G['summary'], 'geometry_table': ov, 'collapse_flags': flags})
    display(ov)
    display(flags)
    run.figure(F.lines(G['spectrum'].group_by('representation', 'component').agg(pl.col('variance_share').mean()), 'component', 'variance_share', 'representation',
                       'Covariance spectrum of held-out embeddings (mean over folds)', 'principal component', 'share of variance', logy=True),
               'c1_covariance_spectrum', {'spectrum': G['spectrum']}, 'How many directions carry variance (collapse check)?', 'Log scale; steep decay = low effective dimension.')
    nr = G['norms'].with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String).alias('rep'))
    run.figure(F.distribution_compare(nr, 'norm', 'domain', 'Embedding norms by domain', 'L2 norm', facet='rep', order=['combine', 'nfl']), 'c1_embedding_norms',
               {'norms_summary': nr.group_by('rep', 'domain').agg(pl.col('norm').median())}, 'Do norms alone reveal the domain?', 'Per representation.')
    if G['pairs'].height:
        pp = G['pairs'].unpivot(index=['representation', 'fold', 'nfl_id'], on=['own_cosine', 'other_same_position_cosine'], variable_name='pair', value_name='cosine')
        pp = pp.with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String).alias('rep'))
        run.figure(F.distribution_compare(pp, 'cosine', 'pair', 'Own-athlete vs other-athlete cross-domain similarity', 'cosine (Combine mean vs NFL mean)', facet='rep'),
                   'c1_athlete_pair_similarity', {'pairs': pp}, 'Is an athlete closer to their own NFL movement than to same-position peers?',
                   'Retrieval contrast as distributions (held-out athletes).')
    sl = G['summary_long']
    dauc = sl.filter(pl.col('metric') == 'domain_auc_grouped')
    run.figure(F.fold_points(dauc, 'representation', 'fold', 'value', 'Held-out domain separability per fold', 'domain AUC (0.5 = not separable)', ref=0.5),
               'c3_domain_auc_by_fold', {'domain_auc': dauc}, 'Can a linear read-out tell Combine from NFL?', 'Athlete-grouped ridge; dots = folds.')
    sc = G['summary'].filter(pl.col('metric').is_in(['mmd2', 'domain_auc_grouped'])).pivot(on='metric', index='representation', values='mean')
    run.figure(F.scatter(sc, 'mmd2', 'domain_auc_grouped', 'MMD² vs domain AUC (held-out rows, mean over folds)', 'MMD² (RBF, median heuristic)', 'domain AUC',
                         label='representation', ref_y=0.5), 'c3_mmd_vs_domain_auc', {'points': sc},
               'Does lower MMD² mean domain invariance?', 'Low MMD² with high AUC = aligned distributions that remain separable.')
    dp = (G['domain_long'].filter((pl.col('metric') == 'domain_auc') & pl.col('value').is_not_nan()).group_by('representation', 'position_group')
          .agg(pl.col('value').mean()).with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String)))
    run.figure(F.heatmap(dp, 'representation', 'position_group', 'value', 'Within-position domain AUC (held-out, mean over folds)', vmin=0.5, vmax=1.0),
               'c3_domain_auc_by_position', {'domain_auc_position': dp}, 'Is separability driven by position mix?', 'Position-adjusted view.')
    PR = ablations.probes_all(exp, MODELS_FOLDS, cfg.seed)
    if PR.height:
        run.table(PR, 'inductive_probes', 'ablations')
        R['probes'] = PR
        pm = PR.group_by('representation', 'domain', 'probe').agg(pl.col('r2_cv').mean()).with_columns(
            (pl.col('domain') + ' · ' + pl.col('probe')).alias('target'), pl.col('representation').map_elements(F.short, return_dtype=pl.String))
        run.figure(F.heatmap(pm, 'representation', 'target', 'r2_cv', 'Physical probes on held-out embeddings (athlete-grouped CV R²)'), 'c1_physical_probes',
                   {'probes': pm}, 'Do embeddings of unseen athletes encode physical quantities?', 'Ridge probes fitted inside the held-out rows.')
    return G

GEOM = run.run_stage('C1', _c, note='geometry, collapse, domain geometry, probes', inputs=['inductive embeddings'], enabled=cfg.run_full_diagnostics)
if GEOM is not None:
    run.stage_begin('C3', 'computed within C1')
    run.stage_end('C3', 'executed', note='computed within C1 (domain geometry)')

In [ ]:
# C4: exploratory projections (fit scope: a seeded, domain-balanced sample of held-out rows of one fold)
def _c4(ctx):
    f = FOLDS[0]
    out = []
    for n in [m for m in (F0N, F2N, F4N) if f in MODELS_FOLDS.get(m, [])]:
        co = geometry.projection_coordinates(exp, n, f, cfg.seed, cfg.max_points_projection, cfg.use_umap)
        run.data_file(co, f'embeddings/inductive/projection_{F.short(n)}_fold{f}', {'seed': cfg.seed, 'fold': f, 'model': n})
        for meth in co['method'].unique().to_list():
            c = co.filter(pl.col('method') == meth)
            run.figure(F.projection(c, f'{F.short(n)} · {meth.upper()} of held-out embeddings (fold {f})'), f'c4_projection_{F.short(n)}_{meth}', {'coords': c},
                       'Exploratory: how are domains and positions arranged?', str(c['fit_scope'][0]), show=meth == 'pca')
        out.append(co)
    return out

run.run_stage('C4', _c4, note='projections', enabled=cfg.run_full_diagnostics)

In [ ]:
lines = []
if GEOM is not None:
    ov = R['geometry_table']
    g_ = lambda r, k: synthesis._f(r.get(k))
    for r in ov.iter_rows(named=True):
        lines.append(f"- {F.short(r['representation'])}: effective rank {g_(r, 'effective_rank'):.2f}, domain AUC {g_(r, 'domain_auc_grouped'):.3f} "
                     f"(position-adjusted {g_(r, 'position_adjusted_domain_auc'):.3f}), MMD² {g_(r, 'mmd2'):.4f}")
    for r in R['collapse_flags'].iter_rows(named=True):
        if r['collapse_flag']:
            lines.append(f"- COLLAPSE FLAG: {F.short(r['representation'])} meets the declared collapse rule.")
    hi = ov.filter(pl.col('domain_auc_grouped') >= 0.9)['representation'].to_list() if 'domain_auc_grouped' in ov.columns else []
    lines.append(f'- Representations whose held-out domains stay linearly separable (AUC ≥ 0.9): {[F.short(x) for x in hi] or "none"}.')
synthesis.section_report(run, 'c_representation_geometry', 'Part C · Representation geometry', 'What do the embeddings encode; are they collapsed; how separable are the domains?',
                         'Distinguishes collapse, domain encoding and missing athlete signal as explanations of near-chance retrieval.',
                         ['held-out rows per fold', 'domain_diagnostics + rep.domain_shift', 'grouped ridge probes'],
                         {'overview': R.get('geometry_table'), 'collapse': R.get('collapse_flags')}, lines,
                         ['Domain AUC is a linear read-out; non-linear separability may be higher.', 'Projections are fitted on a sample and are exploratory.'],
                         status=f"C1 {run.stage_status('C1')} · C4 {run.stage_status('C4')}")
print('\n'.join(lines))

# PART D · Cross-domain retrieval

### Research question
Given an athlete's Combine movement, can a representation pick out the same athlete's NFL movement among same-position peers, **for athletes the encoder never saw**?

### Scientific rationale
This is the most direct test of an athlete-specific, domain-shared representation. Transductive retrieval cannot test generalisation.

### Data and methods
- **D1** reloads the saved *transductive* reference records. They are shown only when the bundle and evaluation level match, and they are never pooled with inductive results.
- **D3/D4** use `Experiment.compare_inductive`. For each outer fold, the query is the athlete's mean Combine embedding. The candidates are the NFL means of the *other held-out athletes of the same fold and position*. These pools depend only on metadata, so every representation sees **identical pools**, and each row carries a `pool_hash` that is checked.
- Athlete AUC is the share of candidates ranked below the own match. Rank metrics (top-1, recall@K, MRR, median rank) are reported with the **chance value of the same pools**.
- Paired Δ uses an athlete bootstrap stratified by fold, with the same draws for every representation. Subgroups with fewer than `min_subgroup` athletes get no interval.

### Interpretation rule
Supported only if the paired Δ interval excludes 0 **and** the mean AUC interval is above 0.5.

In [ ]:
def _d1(ctx):
    tr = environment.transductive_reference(cfg, exp)
    if not tr.height:
        raise StageSkipped('no saved transductive metrics in the reference experiments')
    keep = [c for c in ('source_experiment', 'same_bundle', 'same_science', 'representation', 'level', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi',
                        'cross_n_athletes', 'domain_auc', 'mmd2', 'null_mean', 'source_bundle') if c in tr.columns]
    t = tr.select(keep)
    run.table(t, 'transductive_reference', 'retrieval')
    R['transductive_reference_table'] = t
    display(t)
    ok = t.filter(pl.col('same_bundle') & pl.col('level').is_in(['standard', 'full'])) if 'level' in t.columns else t.filter(pl.col('same_bundle'))
    if ok.height:
        d = ok.with_columns((pl.col('source_experiment') + ' · ' + pl.col('representation') + ' · ' + pl.col('level')).alias('label'))
        run.figure(F.forest(d, 'label', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi', 'Transductive cross-domain retrieval (reloaded records, same bundle)',
                            'retrieval AUC (0.5 = chance)', ref=0.5, ref_label='chance', color_by='representation',
                            subtitle='Transductive diagnostic: all athletes seen in training; not comparable with the inductive results.'),
                   'd1_transductive_reference', {'records': d}, 'Are the earlier transductive numbers reproducible from saved records?', 'Reloaded, not recomputed.')
    else:
        print('No reference record shares this bundle fingerprint: earlier transductive values are NOT comparable (shown for provenance only).')
    ctx['cache'] = 'hit'
    return t

run.run_stage('D1', _d1, note='transductive reference (reloaded)')

In [ ]:
def _d3(ctx):
    global CMP, SCORES, MARGINS
    CMP = exp.compare_inductive(models=MODELS, reference=F0N, bootstrap=cfg.boot, folds=FOLDS, min_subgroup=cfg.min_subgroup, adequate_n=cfg.adequate_n,
                                on_missing='skip', diagnostics=True, save=True, reuse=cfg.reuse_artifacts)
    SCORES = retrieval_lab.extend_scores(CMP.scores)
    run.table(CMP.comparison, 'paired_inductive_comparison', 'retrieval')
    run.table(CMP.subgroups, 'position_subgroups', 'retrieval')
    run.table(CMP.coverage, 'inductive_coverage', 'retrieval')
    run.table(CMP.domain, 'domain_diagnostics_comparison', 'retrieval')
    run.data_file(SCORES, 'embeddings/inductive/athlete_retrieval_scores', {'experiment': exp.id, 'models': MODELS, 'folds': FOLDS})
    RS, RC, BF = retrieval_lab.rank_summary(CMP.scores), retrieval_lab.recall_curve(CMP.scores), retrieval_lab.by_fold(CMP.scores)
    pairs = [(F4N, F0N), (F4N, F2N), (F2N, F0N), (F4N, RANDN), (RANDN, F0N)] + [(m, F0N) for m in ENCODERS if m not in (F2N, F4N)]
    PP = retrieval_lab.paired_pairs(CMP.scores, pairs, n_boot=cfg.boot, seed=cfg.seed, min_n=cfg.min_subgroup, adequate_n=cfg.adequate_n)
    MARGINS = retrieval_lab.margins_all(exp, MODELS_FOLDS)
    for t, n in ((RS, 'rank_summary'), (RC, 'recall_curve'), (BF, 'retrieval_by_fold'), (PP, 'paired_pairs'), (MARGINS, 'similarity_margins')):
        run.table(t, n, 'retrieval')
    R.update({'paired_inductive_comparison': CMP.comparison, 'rank_summary': RS, 'paired_pairs': PP, 'retrieval_by_fold': BF})
    display(CMP.coverage)
    display(CMP.summary())
    display(RS)
    auc = CMP.comparison.filter((pl.col('metric') == 'auc') & (pl.col('status') == 'ok'))
    run.figure(F.forest(auc.with_columns((pl.col('representation').map_elements(F.short, return_dtype=pl.String) + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label')),
                        'label', 'mean', 'mean_lo', 'mean_hi', 'Held-out athlete retrieval (inductive, paired athletes, identical pools)', 'mean athlete AUC',
                        ref=0.5, ref_label='chance', color_by='representation'), 'd3_retrieval_auc', {'comparison': auc},
               'Does any representation retrieve unseen athletes above chance?', 'Fold-stratified athlete bootstrap intervals.')
    run.figure(F.fold_points(BF, 'representation', 'fold', 'auc', 'Retrieval AUC per outer fold', 'mean athlete AUC', ref=0.5), 'd3_retrieval_by_fold', {'by_fold': BF},
               'Is the result consistent across folds?', 'Dots = folds; bar = mean.')
    run.figure(F.lines(RC, 'k', 'recall', 'representation', 'Recall@K vs chance for the realised pools', 'K', 'recall@K', ref='chance', ref_label='chance (same pools)'),
               'd3_recall_at_k', {'recall': RC}, 'How often is the own athlete in the top K?', 'Dotted = chance for the realised pool sizes.')
    run.figure(F.distribution_compare(SCORES.filter(pl.col('status') == 'scored').with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String)),
                                      'normalized_rank', 'representation', 'Normalized rank of the own athlete (0 = top, 1 = bottom)', 'normalized rank'),
               'd3_rank_distribution', {}, 'Where does the own match land in the ranking?', 'Uniform on [0, 1] under chance.')
    if PP.height and 'delta' in PP.columns:
        pd_ = PP.filter(pl.col('metric') == 'auc').with_columns((pl.col('representation').map_elements(F.short, return_dtype=pl.String) + ' − '
                                                                 + pl.col('reference').map_elements(F.short, return_dtype=pl.String)
                                                                 + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
        run.figure(F.forest(pd_, 'label', 'delta', 'delta_lo', 'delta_hi', 'Paired Δ AUC (same athletes, same pools)', 'Δ mean athlete AUC (positive = first better)',
                            ref=0.0, ref_label='no difference', color_by='representation'), 'd4_paired_delta_auc', {'paired': pd_},
                   'Do F4 / F2 beat F0 and each other on the same held-out athletes?', 'Paired fold-stratified bootstrap.')
    sg = CMP.subgroups.filter((pl.col('status') == 'ok') & (pl.col('reliability') != 'insufficient')).with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String))
    if sg.height:
        run.figure(F.heatmap(sg, 'representation', 'position_group', 'mean', 'Retrieval AUC by position (subgroups with ≥ min_subgroup athletes)', diverging=True, center=0.5),
                   'd3_position_heatmap', {'subgroups': sg}, 'Is retrieval concentrated in particular positions?', 'Diverging around chance (0.5); descriptive.')
    if MARGINS.height:
        mg = MARGINS.with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String))
        run.figure(F.distribution_compare(mg, 'margin', 'representation', 'Own similarity minus hardest negative', 'cosine margin (> 0 = own athlete ranked first)'),
                   'd3_positive_negative_separation', {'margins_summary': mg.group_by('representation').agg(pl.col('margin').median(), pl.col('z_own').median())},
                   'How far is the own match from the best impostor?', 'Per held-out athlete.')
    return CMP

CMP = run.run_stage('D3', _d3, note='inductive retrieval + paired comparisons', inputs=['inductive embeddings'], enabled=cfg.run_inductive)
if CMP is not None:
    run.stage_begin('D4', 'computed within D3')
    run.stage_end('D4', 'executed', note='paired comparisons computed within D3')

In [ ]:
def _d5(ctx):
    if CMP is None or not MARGINS.height or F4N not in MODELS_FOLDS:
        raise StageSkipped('needs D3 scores and the F4 margins')
    cs = retrieval_lab.select_case_studies(CMP.scores, MARGINS, F4N, F0N, n=cfg.n_case_studies)
    cs = cs.with_columns(pl.col('nfl_id').map_elements(PSEUDO, return_dtype=pl.String).alias('athlete'),
                         pl.col('top_candidate').map_elements(PSEUDO, return_dtype=pl.String).alias('top_retrieved'))
    run.table(cs, 'case_studies', 'retrieval')
    display(cs.select('case_type', 'athlete', 'position_group', 'rank', 'ref_rank', 'n_candidates', 'margin', 'top_retrieved'))
    for ct in cs['case_type'].unique(maintain_order=True).to_list():
        sub = cs.filter(pl.col('case_type') == ct).head(4)
        labels = {int(r['nfl_id']): f"{r['athlete']} · {r['position_group']} · F4 rank {r['rank']}/{r['n_candidates'] + 1} (F0 {r['ref_rank']})" for r in sub.iter_rows(named=True)}
        prof = retrieval_lab.speed_profiles(exp, list(labels), labels)
        prof = prof.with_columns(pl.col('athlete').map_elements(PSEUDO, return_dtype=pl.String))
        run.figure(F.profile_panels(prof, f'Retrieval case studies · {ct.replace("_", " ")}', 'Mean ± SD speed of Combine forty attempts (blue) and NFL windows (orange); pseudonymised.'),
                   f'd5_case_{ct}', {'profiles': prof}, 'What movement patterns accompany successes and failures?', 'Illustrations selected by explicit rules; not evidence.',
                   show=ct in ('strong_success', 'strong_failure'))
    return cs

run.run_stage('D5', _d5, note='case studies', inputs=['D3'], enabled=cfg.run_inductive)

In [ ]:
lines = []
if CMP is not None:
    c = CMP.comparison.filter((pl.col('metric') == 'auc') & (pl.col('status') == 'ok'))
    for r in c.iter_rows(named=True):
        lines.append(synthesis.describe_interval(f"{F.short(r['representation'])} mean athlete AUC (n={r['n_athletes']})", r['mean'], r['mean_lo'], r['mean_hi'],
                                                 positive_means='above chance', null=0.5))
    for r in R['paired_pairs'].filter(pl.col('metric') == 'auc').iter_rows(named=True) if 'metric' in R['paired_pairs'].columns else []:
        lines.append(synthesis.describe_interval(f"Δ AUC {F.short(r['representation'])} − {F.short(r['reference'])}", r['delta'], r['delta_lo'], r['delta_hi']))
    if len(PRIMARY_FOLDS) < len(ALL_FOLDS):
        lines.append(f'- Only folds {PRIMARY_FOLDS} have an F4 inductive encoder: results are pilot-level and cannot pass Gate 3.')
synthesis.section_report(run, 'd_cross_domain_retrieval', 'Part D · Cross-domain retrieval', 'Can representations retrieve unseen athletes across domains?',
                         'Direct test of an athlete-specific, domain-shared representation.',
                         ['compare_inductive (identical pools, fold-stratified paired bootstrap)', 'rank metrics with pool-size chance baselines', 'transductive records reloaded separately'],
                         {'comparison': R.get('paired_inductive_comparison'), 'rank summary': R.get('rank_summary'), 'paired pairs': R.get('paired_pairs')}, lines,
                         ['Pools within fold × position are small; AUCs are coarse.', 'Five encoders per family are pooled after per-fold scoring.'],
                         status=f"D1 {run.stage_status('D1')} · D3 {run.stage_status('D3')} · D5 {run.stage_status('D5')}")
print('\n'.join(lines))

# PART E · NFL outcome-transfer experiments

### Research question
Do inductive tracking representations improve **out-of-fold** prediction of the frozen NFL movement-capacity target `nfl.q90_shared` beyond the official Combine measurements (M1) and engineered summaries (M1+F0)?

### Scientific rationale
This is the part closest to the competition objective. Better embeddings do not automatically mean better outcome prediction.

### Data and methods
The ladder is `Experiment.transfer_predictions`, which saves out-of-fold predictions for the exact ladder of `Experiment.transfer`:

- athlete outer folds;
- a train-fold-only design (imputation, scaling, position levels);
- inductive embeddings of fold *f* taken from the encoder trained without fold *f*;
- embeddings reduced by a PCA fitted on the training athletes only (≤ 8 components);
- inner athlete-CV for the ridge α.

Every metric below is computed **from the saved predictions**. ΔRMSE = RMSE(reference) − RMSE(model), so positive means the model is better, with a paired athlete bootstrap.

**Cohorts.** *Common*: all ladder athletes. Athletes without an embedding get train-median imputation plus a missing indicator. *Native coverage*: only athletes that have every embedding involved. An encoder enters the ladder only if it was trained on **all** outer folds.

### Interpretation rule
Gate 4 passes only if the ΔRMSE interval is above 0 against **both** M1 and M1+F0.

In [ ]:
def _e2(ctx):
    global OOF
    tm = [m for m in ENCODERS if len(MODELS_FOLDS.get(m, [])) == len(ALL_FOLDS)]
    skipped = [F.short(m) for m in ENCODERS if m not in tm]
    if skipped:
        print('encoders without all outer folds (left out of the ladder):', skipped)
    OOF = exp.transfer_predictions(models=tm, reuse=cfg.reuse_artifacts)
    if tm:
        official = exp.transfer(models=tm, reuse=cfg.reuse_artifacts)
        run.table(official, 'transfer_official_record', 'prediction')
    run.table(OOF, 'oof_predictions', 'prediction')
    models = transfer_lab.ladder_models(OOF)
    PM, FM = transfer_lab.prediction_metrics(OOF), transfer_lab.fold_metrics(OOF)
    TC = transfer_lab.paired_comparisons(OOF, cfg.boot, cfg.seed)
    for t, n in ((PM, 'prediction_metrics'), (FM, 'fold_metrics'), (TC, 'paired_comparisons')):
        run.table(t, n, 'prediction')
    R.update({'prediction_metrics': PM, 'transfer_comparisons': TC})
    display(PM)
    display(TC.select('cohort', 'model', 'reference', 'n_athletes', 'rmse_ref', 'rmse_model', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict'))
    run.figure(F.pred_vs_actual(OOF, models, 'Out-of-fold predictions vs observed nfl.q90_shared'), 'e2_pred_vs_actual', {'oof': OOF.select('nfl_id', 'fold', 'y', *[f'pred_{m}' for m in models])},
               'How closely do the predictions track the outcome?', 'Dashed = identity.')
    d = TC.with_columns((pl.col('model').map_elements(F.short, return_dtype=pl.String) + ' vs ' + pl.col('reference').map_elements(F.short, return_dtype=pl.String)
                         + ' · ' + pl.col('cohort') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    run.figure(F.forest(d, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Paired Δ RMSE (positive = model better)', 'Δ RMSE (target units)', ref=0.0, ref_label='no change',
                        color_by='model'), 'e2_delta_rmse', {'comparisons': d}, 'Does tracking add out-of-fold value beyond M1 and M1+F0?', 'Paired athlete bootstrap.')
    run.figure(F.fold_points(FM, 'model', 'fold', 'rmse', 'Out-of-fold RMSE per outer fold', 'RMSE'), 'e2_fold_variability', {'folds': FM},
               'How variable is performance across folds?', 'Dots = folds.')
    RF = transfer_lab.residual_frame(OOF)
    run.figure(F.residual_panels(RF, models, 'Residual diagnostics'), 'e2_residuals', {'residuals': RF}, 'Where and how do models err?',
               'Left: residual ECDF; right: median residual by observed decile.')
    return OOF

OOF = run.run_stage('E2', _e2, note='transfer ladder (OOF)', inputs=['inductive embeddings (all folds)', 'athletes.parquet'], enabled=cfg.run_transfer,
                    reason_disabled='run_transfer=False')

In [ ]:
def _e5(ctx):
    if OOF is None:
        raise StageSkipped('needs E2 predictions')
    models = transfer_lab.ladder_models(OOF)
    main = f'M1+{F4N}' if f'M1+{F4N}' in models else ('M1+F0' if 'M1+F0' in models else models[-1])
    PE = pl.concat([transfer_lab.position_effects(OOF, m, 'M1', cfg.boot, cfg.seed, cfg.min_subgroup, cfg.adequate_n) for m in models if m != 'M1'], how='diagonal_relaxed')
    run.table(PE, 'position_effects', 'prediction')
    R['position_effects'] = PE.filter(pl.col('model') == main)
    pe = PE.filter(pl.col('reliability') != 'insufficient').with_columns((pl.col('model').map_elements(F.short, return_dtype=pl.String) + ' · ' + pl.col('position_group')
                                                                           + ' (n=' + pl.col('n_athletes').cast(pl.String) + ', ' + pl.col('reliability') + ')').alias('label'))
    run.figure(F.forest(pe, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Position-level Δ RMSE vs M1', 'Δ RMSE (positive = model better)', ref=0.0, color_by='model'),
               'e5_position_effects', {'positions': pe}, 'Do learned features help particular positions?', 'Exploratory below adequate_n; BH column in the table.')
    EA = transfer_lab.error_analysis(OOF, models)
    for k, v in EA.items():
        if v is not None and v.height:
            run.table(v.with_columns(pl.col('nfl_id').map_elements(PSEUDO, return_dtype=pl.String).alias('athlete')) if 'nfl_id' in v.columns else v, f'errors_{k}', 'prediction')
    display(EA['by_capacity'])
    display(EA['residual_vs_officials'])
    SM = transfer_lab.scouting_metrics(OOF, models, 'M1', cfg.boot, cfg.seed)
    run.table(SM['ranking'], 'scouting_ranking', 'prediction')
    run.table(SM['calibration'], 'scouting_calibration', 'prediction')
    run.table(SM['thresholds'], 'scouting_thresholds', 'prediction')
    R['scouting_ranking'] = SM['ranking']
    display(SM['ranking'])
    run.figure(F.calibration(SM['calibration'], 'Decile calibration of out-of-fold predictions'), 'e7_calibration', {'calibration': SM['calibration']},
               'Are predicted capacities calibrated?', 'Mean observed vs mean predicted per prediction decile.')
    run.figure(F.lines(SM['thresholds'], 'threshold_quantile', 'precision', 'model', 'Selection precision vs decision threshold', 'threshold (quantile of target / prediction)',
                       'precision', ref='base_rate', ref_label='base rate'), 'e7_threshold_sensitivity', {'thresholds': SM['thresholds']},
               'How sensitive is top-athlete identification to the threshold?', 'Dotted = share of athletes above the threshold.')
    if f'pred_{main}' in OOF.columns:
        run.figure(F.scatter(OOF, 'pred_M1', f'pred_{main}', f'Ranking stability: M1 vs {F.short(main)} predictions', 'M1 prediction', f'{F.short(main)} prediction', identity=True),
                   'e7_ranking_stability', {'oof': OOF.select('nfl_id', 'pred_M1', f'pred_{main}')}, 'How much does tracking change athlete rankings?', 'Dashed = identity.')
        top = OOF.filter(pl.col('y') >= pl.col('y').quantile(0.9)).with_columns(pl.col('nfl_id').map_elements(PSEUDO, return_dtype=pl.String).alias('athlete'))
        run.table(top, 'high_capacity_case_studies', 'prediction')
        display(top.select('athlete', 'position_group', 'y', *[c for c in top.columns if c.startswith('pred_')]).head(15))
    return PE

run.run_stage('E5', _e5, note='positions, errors, scouting value', inputs=['E2'], enabled=cfg.run_transfer, reason_disabled='run_transfer=False')
if run.stage_ok('E5'):
    run.stage_begin('E7', 'computed within E5')
    run.stage_end('E7', 'executed', note='scouting metrics computed within E5')

if cfg.run_permutation_tests and OOF is not None and len(MODELS_FOLDS.get(F4N, [])) == len(ALL_FOLDS):
    PT = run.run_stage('E_perm', lambda ctx: transfer_lab.permutation_test(exp, 'F4', exp.cfg.science.n_perm, cfg.seed), note='within-position permutation null')
    if PT is not None:
        run.table(pl.DataFrame([PT]), 'permutation_test', 'prediction')
        print(PT)

In [ ]:
lines = []
if R.get('transfer_comparisons') is not None:
    for r in R['transfer_comparisons'].iter_rows(named=True):
        lines.append(synthesis.describe_interval(f"Δ RMSE {F.short(r['model'])} vs {F.short(r['reference'])} [{r['cohort']}, n={r['n_athletes']}]", r['delta_rmse'], r['delta_lo'], r['delta_hi'],
                                                 positive_means='model better'))
else:
    lines.append('- Transfer ladder not executed in this run (run_transfer=False or no encoder trained on all folds).')
synthesis.section_report(run, 'e_nfl_outcome_transfer', 'Part E · NFL outcome transfer', 'Do inductive tracking representations improve out-of-fold prediction of nfl.q90_shared?',
                         'Closest test of the competition objective.', ['Milestone 12 ridge ladder, train-fold-only preprocessing', 'paired athlete bootstrap Δ RMSE', 'common and native-coverage cohorts'],
                         {'metrics': R.get('prediction_metrics'), 'comparisons': R.get('transfer_comparisons'), 'scouting': R.get('scouting_ranking')}, lines,
                         ['Decision metrics are descriptive and make no coaching-impact claim.', 'The 12C target estimator was frozen on NFL data of all athletes before any Combine join.'],
                         status=f"E2 {run.stage_status('E2')} · E5 {run.stage_status('E5')}")
print('\n'.join(lines))

# PART F · Representation objective ablations

### Research question
Why does F4 behave as it does? Which loss terms matter?

### Hypotheses
1. Physics supervision improves physical validity (`F4_nophys`, `F4_shuffled_phys`).
2. Contrastive alignment improves athlete specificity (`F4_noalign`).
3. MMD-style alignment lowers distribution distance without lowering domain AUC (`F4_mmd`, `F4_adv`).
4. Alignment by category vs matched state changes what is preserved (`F4_category`).

### Methods
Ablation names are read from the registry at runtime: `reg.ablations()`, which is `se.ABLATIONS`. Each ablation is trained inductively on the same folds as F4 (`run_mode='full'`, `run_ablations=True`, `run_training=True`; adopted checkpoints count).

Each ablation is compared with F4 on the **same** held-out athletes and pools (paired Δ AUC), using the Part C diagnostics and the probes. The summary table lists **benefits and regressions** under declared rules. Model complexity (parameters, saved training time) is reported for the Pareto view in Part J.

In [ ]:
def _f(ctx):
    ABL = ablations.ablation_names()
    print('registry ablations:', ABL)
    if cfg.run_training and cfg.run_ablations:
        environment.train_missing(exp, ABL, FOLDS, resume=cfg.resume)
    mf = environment.available_pairs(exp, [F4N, *ABL], FOLDS)
    avail = [a for a in ABL if mf.get(a)]
    if not avail:
        raise StageSkipped('no ablation encoder trained or adopted (set run_mode=full, run_ablations=True, run_training=True)')
    comp = exp.compare_inductive(models=[F4N, *avail], reference=F4N, bootstrap=cfg.boot, folds=FOLDS, min_subgroup=cfg.min_subgroup, adequate_n=cfg.adequate_n,
                                 on_missing='skip', diagnostics=False, save=False, reuse=cfg.reuse_artifacts)
    sub_mf = {k: v for k, v in mf.items() if k in avail}
    G = geometry.run_geometry(exp, {**sub_mf, F4N: mf.get(F4N, [])}, cfg.seed)
    PRa = ablations.probes_all(exp, sub_mf, cfg.seed)
    parts_ = [x for x in (R.get('probes'), PRa) if x is not None and x.height]
    allp = pl.concat(parts_, how='diagonal_relaxed').unique(subset=['representation', 'fold', 'domain', 'probe'], keep='last') if parts_ else pl.DataFrame()
    if allp.height:
        run.table(allp, 'inductive_probes', 'ablations')
        R['probes'] = allp
    S = ablations.ablation_summary(comp.comparison, G['summary'], allp, F4N)
    run.table(comp.comparison, 'ablation_paired_comparison', 'ablations')
    run.table(G['summary'], 'ablation_geometry', 'ablations')
    run.table(S, 'ablation_summary', 'ablations')
    R['ablation_summary'] = S
    display(S)
    d = comp.comparison.filter((pl.col('metric') == 'auc') & (pl.col('representation') != F4N) & (pl.col('status') == 'ok'))
    run.figure(F.forest(d, 'representation', 'delta', 'delta_lo', 'delta_hi', 'Ablations: paired Δ AUC vs F4 (same athletes and pools)', 'Δ mean athlete AUC (positive = ablation better)',
                        ref=0.0, ref_label='F4'), 'f_ablation_delta_auc', {'ablations': d}, 'Which loss terms matter for retrieval?', 'Paired bootstrap vs F4.')
    hm = G['summary'].filter(pl.col('metric').is_in(['domain_auc_grouped', 'mmd2', 'effective_rank', 'knn_domain_mixing_excess'])).select('representation', 'metric', 'mean')
    run.figure(F.heatmap(hm, 'representation', 'metric', 'mean', 'Ablation geometry (held-out, mean over folds)'), 'f_ablation_geometry', {'geometry': hm},
               'How does each objective change domain geometry and rank?', 'Read with the benefit / regression columns.')
    cl = ablations.curves_long(ablations.training_curves(exp, [F4N, *avail]))
    if cl.height:
        run.figure(F.small_multiples_lines(cl, 'component', 'epoch', 'value', 'run', 'Ablation loss trajectories', 'epoch', 'loss', max_facets=9), 'f_ablation_loss_trajectories',
                   {'curves': cl}, 'Do ablations train differently?', 'Per loss term.')
    return S

run.run_stage('F1', _f, note='objective ablations', inputs=['ablation encoders'], enabled=cfg.run_mode == 'full' or bool(cfg.run_ablations),
              reason_disabled='ablations run in full mode only')

CX = ablations.complexity_table(exp, [F0N, RANDN, *ENCODERS, *[a for a in ablations.ablation_names() if environment.available_pairs(exp, [a], FOLDS).get(a)]],
                                ablations.training_curves(exp))
run.table(CX, 'complexity', 'ablations')
R['complexity'] = CX
display(CX)

# PART G · Sensitivity and robustness

### Research question
Do the retrieval conclusions survive harder candidate pools, sequence-quality filters, subgroups, training seeds and less NFL data?

### Methods
- **G3 pools.** Pools are defined only from metadata and official measurements, so they are identical for every representation: random same-position sub-pools of size k; **stopwatch twins**, the k same-position athletes nearest in official forty + weight (can movement distinguish athletes the stopwatch cannot?); and a cross-position pool. Chance is reported for each pool.
- **G4 quality.** Held-out rows are filtered by valid frames, heading quality, extreme speed or acceleration, and NFL window duration. Pools change with a filter, but identically for every representation.
- **G5 heterogeneity.** Paired Δ within draft year, M1 completeness, NFL coverage tercile and Combine speed tercile. Descriptive only.
- **G1 seeds / G2 fractions.** GPU work, opt-in. Seeds are separate experiments. Fractions are *transductive* diagnostics only (API limit).

In [ ]:
ROB_MF = {k: v for k, v in MODELS_FOLDS.items() if k in (F0N, F2N, F4N, RANDN)}

def _g3(ctx):
    sc, comp = retrieval_lab.pool_difficulty(exp, ROB_MF, F0N, cfg.seed, cfg.boot)
    run.data_file(sc, 'embeddings/inductive/pool_scores', {'models': list(ROB_MF), 'seed': cfg.seed})
    run.table(comp, 'pool_comparison', 'robustness')
    R['pool_comparison'] = comp
    d = comp.filter((pl.col('metric') == 'auc') & (pl.col('representation') != F0N) & (pl.col('status') == 'ok')).with_columns(
        (pl.col('pool') + ' · ' + pl.col('representation').map_elements(F.short, return_dtype=pl.String) + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    run.figure(F.forest(d, 'label', 'delta', 'delta_lo', 'delta_hi', 'Δ AUC vs F0 under alternative candidate pools', 'Δ mean athlete AUC', ref=0.0, color_by='representation'),
               'g3_pool_difficulty', {'pools': d}, 'Does the advantage hold against stopwatch twins and smaller pools?', 'Chance AUC = 0.5 under every pool.')
    display(comp.filter(pl.col('metric') == 'top1').select('pool', 'representation', 'n_athletes', 'mean', 'chance'))
    return comp

run.run_stage('G3', _g3, note='candidate-pool difficulty', enabled=cfg.run_inductive and bool(ROB_MF.get(F0N)))

In [ ]:
def _g4(ctx):
    sc, comp = retrieval_lab.quality_sensitivity(exp, SEQ, ROB_MF, F0N, cfg.seed, cfg.boot)
    run.data_file(sc, 'embeddings/inductive/quality_scores', {'models': list(ROB_MF)})
    run.table(comp, 'quality_comparison', 'robustness')
    R['quality_comparison'] = comp
    d = comp.filter((pl.col('representation') != F0N) & (pl.col('status') == 'ok')).with_columns(
        (pl.col('filter') + ' · ' + pl.col('representation').map_elements(F.short, return_dtype=pl.String) + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    run.figure(F.forest(d, 'label', 'delta', 'delta_lo', 'delta_hi', 'Δ AUC vs F0 under sequence-quality filters', 'Δ mean athlete AUC', ref=0.0, color_by='representation'),
               'g4_quality_sensitivity', {'quality': d}, 'Do results depend on short, extreme or low-quality sequences?', 'A sign flip = fragile.')
    return comp

run.run_stage('G4', _g4, note='sequence-quality sensitivity', enabled=cfg.run_inductive and cfg.run_full_diagnostics and cfg.run_mode != 'pilot',
              reason_disabled='standard / full mode with run_full_diagnostics')

In [ ]:
def _g5(ctx):
    if CMP is None:
        raise StageSkipped('needs D3 scores')
    athf = biomech.athlete_features(SEQ)
    groups = retrieval_lab.heterogeneity_groups(ATH, athf)
    names = [m for m in (F0N, F2N, F4N) if m in ROB_MF]
    parts = [stats.paired_in_subgroups(CMP.scores, names, F0N, groups, v, 'auc', cfg.boot, cfg.seed, cfg.min_subgroup, cfg.adequate_n)
             for v in ('draft_year', 'm1_complete', 'nfl_coverage', 'combine_speed') if v in groups.columns]
    H = pl.concat([p for p in parts if p.height], how='diagonal_relaxed') if any(p.height for p in parts) else pl.DataFrame()
    run.table(H, 'heterogeneity', 'robustness')
    if H.height:
        d = H.filter((pl.col('representation') != F0N) & (pl.col('reliability') != 'insufficient')).with_columns(
            (pl.col('subgroup_variable') + '=' + pl.col('subgroup') + ' · ' + pl.col('representation').map_elements(F.short, return_dtype=pl.String)
             + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
        run.figure(F.forest(d, 'label', 'delta', 'delta_lo', 'delta_hi', 'Δ AUC vs F0 within athlete subgroups (descriptive)', 'Δ mean athlete AUC', ref=0.0, color_by='representation'),
                   'g5_heterogeneity', {'subgroups': d}, 'Are effects consistent across draft years, completeness, coverage and speed profiles?', 'No multiplicity adjustment; descriptive.')
    return H

run.run_stage('G5', _g5, note='population heterogeneity', enabled=cfg.run_inductive and cfg.run_full_diagnostics and cfg.run_mode != 'pilot',
              reason_disabled='standard / full mode with run_full_diagnostics')

In [ ]:
def _g1(ctx):
    if CMP is None:
        raise StageSkipped('needs D3 scores of the main run')
    ex = robustness.seed_scores(cfg, run, F4N, cfg.extra_seeds, FOLDS, train=cfg.run_training)
    if not ex.height:
        raise StageSkipped('no extra-seed encoder available (run_training=True trains them)')
    sc = robustness.seed_comparison(CMP.scores, ex, F4N, cfg.boot, cfg.seed)
    run.table(sc, 'seed_comparison', 'robustness')
    R['seed_comparison'] = sc
    display(sc)
    run.figure(F.forest(sc.filter(pl.col('status') == 'ok'), 'representation', 'mean', 'mean_lo', 'mean_hi', 'F4 retrieval across training seeds (same athletes, same pools)',
                        'mean athlete AUC', ref=0.5, ref_label='chance'), 'g1_seed_sensitivity', {'seeds': sc}, 'Is F4 stable across training seeds?', 'Each seed = separate experiment.')
    return sc

run.run_stage('G1', _g1, note='seed sensitivity', enabled=cfg.run_mode == 'full' and cfg.run_seeds, reason_disabled='full mode with run_seeds=True')

def _g2(ctx):
    lc = robustness.learning_curve(exp, F4N, exp.cfg.science.learning_curve_fractions, train=cfg.run_training)
    if not lc.height:
        raise StageSkipped('no learning-curve run available')
    run.table(lc, 'learning_curve', 'robustness')
    R['learning_curve'] = lc
    run.figure(F.lines(lc.with_columns(pl.lit(F4N).alias('model')), 'nfl_fraction_requested', 'cross_auc', 'model', 'Transductive retrieval vs share of NFL windows (diagnostic)',
                       'share of NFL windows used in training', 'cross-domain AUC', lo='cross_auc_lo', hi='cross_auc_hi', ref=0.5),
               'g2_learning_curve', {'lc': lc}, 'Does more unlabeled NFL data help alignment?', 'Transductive regime only: not data efficiency for unseen athletes.')
    return lc

run.run_stage('G2', _g2, note='NFL-fraction learning curve (transductive)', enabled=cfg.run_mode == 'full' and cfg.run_learning_curves,
              reason_disabled='full mode with run_learning_curves=True')

# PART H · Physics and biomechanics interpretation

### Research question
Which interpretable movement properties, measured at the Combine, are **repeatable**, **transfer** to NFL movement, and **add predictive value beyond M1**?

### Metric definitions
Units are yd, s and rad, restored from the store scaling. Only valid frames are used. All metrics are deterministic, so no fold leakage is possible. The full table is `tables/biomechanics/metric_definitions`.

| metric | expression | reading | undefined / unreliable when |
|---|---|---|---|
| accel_rate_0_90 | (0.9·v_max − v₀) / (t at 0.9·v_max − t₀) | acceleration efficiency from rest | start ≥ 1 yd/s |
| time_to_90 | t at 0.9·v_max − t₀ | build-up time | as above |
| peak_accel / peak_decel | max a_tan / −min a_tan | explosiveness / braking capacity | < 3 frames |
| braking_impulse | Σ max(−a_tan, 0)·Δt | speed removed by braking | < 3 frames |
| turn_at_speed | max abs(a_n) at v ≥ 4 yd/s | speed-dependent turning | never at speed |
| curvature_p90 | q90 of abs(a_n)/v² at v ≥ 2 yd/s | path tightness | < 5 moving frames |
| heading_rate_sd | SD of yaw rate | heading variability | heading valid < 50 % |
| burst_count | runs of a_tan ≥ 3 yd/s² lasting ≥ 0.3 s | explosive bursts | < 3 frames |
| jerk_rms | RMS of Δa_tan/Δt | (un)smoothness; noise-sensitive | < 4 frames |
| turn_asymmetry | (max a_n⁺ − max a_n⁻)/(sum) | left / right imbalance | **sign convention unverified: exploratory** |

These are measured kinematic quantities. None is claimed to represent an anatomical trait.

### Methods
1. **Repeatability:** first vs second attempt of the same drill (Combine), and odd vs even NFL windows. Spearman–Brown corrected, with an athlete bootstrap.
2. **Cross-domain stability:** Spearman of each Combine metric with the same metric in NFL windows, raw and partial on forty + weight.
3. **Incremental value:** ladder M1 vs M1 + metric with the Part E machinery, a paired bootstrap ΔRMSE, and BH across metrics.
4. **Pre-declared hypotheses H1–H6:** partial Spearman with the frozen target, controlling for the stated stopwatch / body covariates, with an athlete bootstrap and BH. These are correlational only.

NFL-window metrics are outcome-side, and they never enter a predictor frame.

In [ ]:
def _h(ctx):
    global ATHF, HDATA
    run.table(biomech.metric_definitions(), 'metric_definitions', 'biomechanics')
    ATHF = biomech.athlete_features(SEQ)
    run.data_file(ATHF, 'features/athlete/athlete_movement_features', {'bundle': exp.bundle.fingerprint, 'version': biomech.VERSION})
    RP = biomech.repeatability(SEQ, max(200, cfg.boot // 2), cfg.seed)
    run.table(RP, 'repeatability', 'biomechanics')
    HDATA = biomech.ladder_frame(exp, ATHF)
    CD = biomech.cross_domain_stability(ATHF.join(exp.bundle.athletes.select('nfl_id', 'forty', 'weight'), on='nfl_id', how='left') if exp.bundle.athletes is not None else ATHF,
                                        max(200, cfg.boot // 2), cfg.seed)
    run.table(CD, 'cross_domain_stability', 'biomechanics')
    R.update({'repeatability': RP, 'cross_domain_stability': CD})
    display(RP.select('metric', 'domain', 'scope', 'n_athletes', 'reliability', 'rel_lo', 'rel_hi'))
    rp = RP.filter(pl.col('reliability').is_not_null() & pl.col('reliability').is_not_nan()).with_columns((pl.col('metric') + ' · ' + pl.col('scope') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    run.figure(F.forest(rp, 'label', 'reliability', 'rel_lo', 'rel_hi', 'Split-half reliability (Spearman–Brown)', 'reliability', ref=0.5, ref_label='0.5 floor', color_by='domain'),
               'h_repeatability', {'reliability': rp}, 'Which movement metrics are repeatable?', 'Combine: attempt 1 vs 2; NFL: odd vs even windows.')
    cd = CD.filter(pl.col('spearman').is_not_nan()).with_columns((pl.col('metric') + ' · ' + pl.col('combine_scope') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'),
                                                                pl.lit('nfl').alias('domain'))
    run.figure(F.forest(cd, 'label', 'spearman', 'lo', 'hi', 'Combine metric vs the same metric in NFL windows', 'Spearman (athlete level)', ref=0.0, color_by='domain'),
               'h_cross_domain_stability', {'stability': cd}, 'Do Combine movement traits persist in NFL movement?', 'Partial (forty + weight) in the table.')
    if HDATA is None:
        raise StageSkipped('blocked: athletes.parquet / frozen target missing — incremental value not evaluable')
    mets = [c for c in HDATA.columns if c.startswith('c_') and not c.endswith('__n_seq')]
    IV = biomech.incremental_value(exp, HDATA, mets, cfg.seed, n_boot=cfg.boot)
    HT = biomech.hypothesis_tests(HDATA, cfg.boot, cfg.seed)
    TW = biomech.forty_twins_dispersion(HDATA)
    for t, n in ((IV, 'incremental_value'), (HT, 'hypothesis_tests'), (TW, 'forty_twins_dispersion')):
        run.table(t, n, 'biomechanics')
    R.update({'incremental_value': IV, 'hypothesis_tests': HT})
    if IV.height:
        iv = IV.with_columns((pl.col('metric') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ', BH ' + pl.col('p_boot_le0_bh').round(3).cast(pl.String) + ')').alias('label'))
        run.figure(F.forest(iv, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Incremental value of each Combine movement metric beyond M1', 'Δ RMSE (positive = metric helps)', ref=0.0),
                   'h_incremental_value', {'incremental': iv}, 'Which interpretable metrics add out-of-fold predictive value?', 'Paired athlete bootstrap; BH across metrics.')
    ht = HT.filter(pl.col('status') == 'evaluated') if 'status' in HT.columns else HT
    if ht.height:
        run.figure(F.forest(ht.with_columns((pl.col('hypothesis') + ' · ' + pl.col('metric')).alias('label')), 'label', 'partial_spearman', 'lo', 'hi',
                            'Pre-declared hypotheses: partial Spearman with nfl.q90_shared', 'partial Spearman (controls in table)', ref=0.0),
                   'h_hypotheses', {'hypotheses': ht}, 'Do braking, curved acceleration, build-up, smoothness or consistency relate to NFL capacity beyond the stopwatch?',
                   'Correlational; BH-adjusted bootstrap p in the table.')
    if 'c_forty__accel_rate_0_90' in HDATA.columns:
        run.figure(F.scatter(HDATA, 'forty', 'c_forty__accel_rate_0_90', 'Same forty, different acceleration build-up?', 'official forty (s)', 'accel rate 0→90 % (yd/s²)',
                             group='position_group'), 'h_forty_twins', {'twins': TW}, 'Can athletes with similar forty times differ in acceleration profile?',
                   'Vertical spread at a given forty = profile differences the stopwatch misses.')
    return IV

run.run_stage('H1', _h, note='biomechanics', inputs=['bundle', 'athletes.parquet'])

# PART I · Interpretability and athlete-level case studies

### Research question
Which inputs carry the out-of-fold transfer signal? What does a research profile of an individual athlete look like?

### Methods
The interpretable model is the same ridge ladder on M1 + Combine movement metrics with coverage ≥ 50 %. Everything is out-of-fold:

- **Permutation importance:** test-fold values are permuted among the test athletes. With one row per athlete, this is athlete-grouped. Reported overall and per position.
- **Partial dependence:** test rows are set to training-fold quantiles (q05–q95), i.e. physically observed values.
- **Interaction test:** metric × forty.
- **Counterfactual sensitivity:** each metric is moved to its within-position q75 in the athlete's own held-out-fold model.

Importance is predictive, not causal, and correlated inputs share importance. **Profiles** are exported to HTML and Markdown under `reports/athlete_profiles/`, with salted pseudonyms. They are exploratory research views, not personnel recommendations.

In [ ]:
def _i(ctx):
    if globals().get('HDATA') is None:
        raise StageSkipped('needs Part H (ladder frame with the frozen target)')
    cand = [c for c in HDATA.columns if c.startswith('c_') and not c.endswith('__n_seq')]
    mets = interpret.choose_metrics(HDATA, cand)
    IL = interpret.interpretable_ladder(HDATA, mets, exp, cfg.seed, n_repeats=20 if cfg.run_mode != 'pilot' else 5)
    run.table(IL['importance'], 'permutation_importance', 'interpretability')
    run.table(IL['pdp'], 'partial_dependence', 'interpretability')
    run.table(IL['coefficients'], 'coefficients_by_fold', 'interpretability')
    run.table(IL['oof'], 'interpretable_oof', 'interpretability')
    R['importance'] = IL['importance']
    print(f"interpretable ladder OOF RMSE: {IL['rmse']:.4f} (features: {len(IL['features'])})")
    imp = IL['importance'].filter(pl.col('position_group') == 'ALL').sort('importance_delta_rmse', descending=True).head(20)
    run.figure(F.hbar(imp, 'feature', 'importance_delta_rmse', 'Permutation importance (Δ RMSE when the input is permuted, out-of-fold)', 'Δ RMSE', fmt='{:+.4f}'),
               'i_permutation_importance', {'importance': imp}, 'Which inputs carry the predictive signal?', 'Athlete-grouped permutation within test folds.')
    pi_ = IL['importance'].filter((pl.col('position_group') != 'ALL') & pl.col('feature').is_in(imp['feature'].head(8).to_list()))
    if pi_.height:
        run.figure(F.heatmap(pi_, 'feature', 'position_group', 'importance_delta_rmse', 'Position-conditioned importance (Δ RMSE)', fmt='{:+.3f}', diverging=True),
                   'i_position_importance', {'importance_position': pi_}, 'Do different positions rely on different inputs?', 'Descriptive; small groups are noisy.')
    top = [f for f in imp['feature'].to_list() if f.startswith('c_')][:6]
    pdp = IL['pdp'].filter(pl.col('feature').is_in(top + ['forty']))
    if pdp.height:
        run.figure(F.small_multiples_lines(pdp.with_columns(pl.lit('mean prediction').alias('s')), 'feature', 'value', 'mean_prediction', 's', 'Feature-response curves (partial dependence, out-of-fold)',
                                           'feature value (physical units)', 'predicted nfl.q90_shared', ncols=3, max_facets=9),
                   'i_feature_response', {'pdp': pdp}, 'How does predicted capacity respond to each movement metric?', 'Grid = training-fold quantiles q05–q95.')
    INT = pl.concat([interpret.interaction_test(HDATA, m, exp, cfg.seed) for m in top[:3]], how='diagonal_relaxed') if top else pl.DataFrame()
    run.table(INT, 'interaction_tests', 'interpretability')
    src = OOF if globals().get('OOF') is not None else IL['oof']
    sel = interpret.select_profiles(src, cfg.n_profiles, cfg.seed)
    res_pos = IL['oof'].with_columns((pl.col('y') - pl.col('pred_interpretable')).alias('r'))
    feats = [*IL['features']]
    index = []
    for row in sel.iter_rows(named=True):
        a = int(row['nfl_id'])
        lab = PSEUDO(a)
        arow = HDATA.filter(pl.col('nfl_id') == a).row(0, named=True)
        preds = {k.removeprefix('pred_'): row[k] for k in row if k.startswith('pred_')}
        il = IL['oof'].filter(pl.col('nfl_id') == a)
        if il.height:
            preds['interpretable (M1 + movement metrics)'] = il['pred_interpretable'][0]
        same = res_pos.filter((pl.col('position_group') == arow['position_group']) & (pl.col('nfl_id') != a))['r'].drop_nans().to_numpy()
        main = preds.get(f'M1+{F4N}', preds.get('interpretable (M1 + movement metrics)'))
        interval = (main + float(np.quantile(same, 0.1)), main + float(np.quantile(same, 0.9))) if same.size >= 10 and main is not None else (float('nan'), float('nan'))
        cf = interpret.counterfactuals(HDATA, IL, [a], exp, cfg.seed, top)
        nb = interpret.neighbours(HDATA, a, feats).with_columns(pl.col('nfl_id').map_elements(PSEUDO, return_dtype=pl.String).alias('label'))
        pos = HDATA.filter(pl.col('position_group') == arow['position_group'])
        pct = {}
        for m in top:
            v = arow.get(m)
            col = pos[m].drop_nulls().to_numpy()
            pct[m] = float((col < v).mean() * 100) if v is not None and col.size else float('nan')
        ar = ATH.filter(pl.col('nfl_id') == a)
        cov = {'Combine sequences': int(ar['n_combine_seq'][0]) if ar.height else None, 'NFL windows': int(ar['n_nfl_seq'][0]) if ar.height else None,
               'position group': arow['position_group'], 'outer fold': arow['fold']}
        md = interpret.athlete_profile(a, lab, row['selection'], arow, preds, row.get('y'), interval, cf, nb, pct, cov)
        run.write_text(md, f'reports/athlete_profiles/{lab}.md')
        run.write_text(markdown_to_html(md, lab), f'reports/athlete_profiles/{lab}.html')
        index.append({'athlete': lab, 'selection': row['selection'], 'position_group': arow['position_group'], 'profile': f'reports/athlete_profiles/{lab}.html'})
    idx = pl.DataFrame(index)
    run.table(idx, 'athlete_profile_index', 'interpretability')
    display(idx)
    return IL

run.run_stage('I1', _i, note='interpretability + athlete profiles', inputs=['H1', 'E2 (optional)'], enabled=cfg.run_mode != 'pilot' or cfg.run_transfer,
              reason_disabled='pilot mode without run_transfer')

# PART J · Integrated scientific comparison and research gates

Every saved table of this run is reloaded (`synthesis.collect`), so this part also works in a fresh session. The gates are evaluated by the declared rules in `synthesis.GATE_RULES`. **Missing evidence is `not_evaluated`, never a pass.**

The integrated table lists objective measurements per representation. The column `researcher_judgment (rule-based)` is judgment produced by stated rules, and is kept separate from the measurements. No weighted ranking is computed.

In [ ]:
def _j(ctx):
    global GATES, INTEGRATED, RS2
    RS2 = synthesis.collect(run)
    GATES = synthesis.evaluate_gates(RS2, primary=F4N, reference=F0N, all_folds=len(ALL_FOLDS))
    INTEGRATED = synthesis.integrated_comparison(RS2, GATES)
    run.table(GATES, 'research_gates', 'synthesis')
    run.table(INTEGRATED, 'integrated_comparison', 'synthesis')
    run.table(pl.DataFrame({'gate': list(synthesis.GATE_RULES), 'rule': list(synthesis.GATE_RULES.values())}), 'gate_rules', 'synthesis')
    display(GATES)
    display(INTEGRATED)
    pts = INTEGRATED.filter(pl.col('parameters_total').is_not_null()) if 'parameters_total' in INTEGRATED.columns else pl.DataFrame()
    if pts.height and 'delta_auc_vs_ref' in pts.columns:
        pts = pts.with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String).alias('label'))
        run.figure(F.scatter(pts, 'parameters_total', 'delta_auc_vs_ref', 'Predictive value vs complexity', 'parameters (encoder + heads)', 'Δ AUC vs F0 (inductive)',
                             label='label', ref_y=0.0), 'j_pareto_complexity', {'points': pts}, 'Is added complexity buying retrieval?', 'Pareto view; no weighted score.')
    if 'delta_rmse_vs_M1' in INTEGRATED.columns:
        tr = INTEGRATED.filter(pl.col('delta_rmse_vs_M1').is_not_null()).with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String).alias('label'))
        if tr.height and 'inductive_auc' in tr.columns:
            run.figure(F.scatter(tr, 'inductive_auc', 'delta_rmse_vs_M1', 'Retrieval vs NFL transfer value', 'inductive retrieval AUC', 'Δ RMSE vs M1', label='label', ref_x=0.5, ref_y=0.0),
                       'j_retrieval_vs_transfer', {'points': tr}, 'Do better embeddings mean better NFL prediction?', 'Each point = representation.')
    return GATES

run.run_stage('J1', _j, note='integrated comparison + gates', inputs=['all saved tables'])

# PART K · Final research report and exports

`reports/markdown/FINAL_COMPREHENSIVE_RESEARCH_REPORT.md` (with an HTML copy) is generated from the saved tables, the gates and the stage ledger only. Each section states whether its content is **measured**, **partially measured** or **not yet executed**.

This part also exports the registry status, the figure index, the artifact inventory and a ZIP archive.

In [ ]:
def _k(ctx):
    status = registry.status_table(run)
    run.save_table(status, 'experiment_registry', subdir='config')
    run.write_text(registry.matrix_markdown(status), 'config/experiment_registry.md')
    rs = synthesis.collect(run)
    gates = globals().get('GATES')
    if gates is None:
        gates = synthesis.evaluate_gates(rs, primary=F4N, reference=F0N, all_folds=len(ALL_FOLDS))
    integ = globals().get('INTEGRATED')
    text = synthesis.final_report(run, rs, gates, integ, status, cfg, env, exp)
    run.save_markdown(text, 'FINAL_COMPREHENSIVE_RESEARCH_REPORT')
    figs = pl.DataFrame(run.results.get('_figures', []), infer_schema_length=None) if run.results.get('_figures') else pl.DataFrame()
    if figs.height:
        run.table(figs, 'figure_index', 'synthesis')
    exp.export(dest=run.root / 'exports' / f'{exp.id}_experiment.zip', include_embeddings=False) if not (run.root / 'exports' / f'{exp.id}_experiment.zip').exists() else None
    inv = run.inventory()
    run.table(inv, 'artifact_inventory', 'synthesis')
    z = run.export_zip(include=('config', 'manifests', 'tables', 'figures', 'reports', 'diagnostics', 'logs'))
    display(status.select('id', 'title', 'status', 'note'))
    display(inv.group_by('area').agg(pl.len().alias('files'), (pl.col('bytes').sum() / 2 ** 20).round(1).alias('MB')).sort('area'))
    print('final report:', run.root / 'reports' / 'markdown' / 'FINAL_COMPREHENSIVE_RESEARCH_REPORT.md')
    print('archive:', z)
    return text

run.run_stage('K', _k, note='final report + exports')
display(run.stage_table())

## Saved artifacts and resuming

Everything is under `MyDrive/NFLBowl/experimental_lab/m12_comprehensive_lab_v1/`:

- `config/`: run configuration, environment, precision lock, experiment registry.
- `datasets/`: audits and cached per-sequence and per-athlete data.
- `features/`: athlete movement features.
- `embeddings/`: athlete scores, projections, norms.
- `checkpoints/experiments/m12_comprehensive_lab_v1/`: the nflbowl experiment (best.pt, epoch shards, optimizer, scaler and RNG state, embeddings, immutable evaluation records).
- `tables/<family>/`: CSV and Parquet.
- `figures/png|svg|pdf/` with `figures/data/`.
- `reports/markdown|html|athlete_profiles/`.
- `diagnostics/`, `logs/`.
- `manifests/`: `stages.json` (stage ledger), `artifacts.jsonl`, `artifact_index.json`.
- `exports/`.

To resume after a disconnect, re-run Setup 1–3 with the same `experiment_name` and the same precision. Then run the sections you need. Saved artifacts are reloaded when their bundle fingerprint and science hash match. Earlier experiments (`f4_main`, `f4_main_clean`, `m12_unified_clean`) are only read, never written.